# Practical 08: Columnar Storage with Apache Parquet Format in PySpark

**UNIT 2: Big Data Processing using PySpark**

**Problem Statement:** Write a PySpark program to demonstrate high-performance columnar storage using Apache Parquet. Create a PySpark DataFrame, export it to Parquet format using .write.mode('overwrite').parquet(), reload the Parquet file using spark.read.parquet(), and verify schema preservation and row counts.

In [1]:
import os
import shutil
import tempfile
from pyspark.sql import SparkSession, Row

sc = (SparkSession.builder.appName("DS505-Pr08").master("local[*]")
         .config("sc.ui.showConsoleProgress", "false").getOrCreate())
sc.sparkContext.setLogLevel("ERROR")

data = [
    {"Name": "Aryan", "Department": "IT", "City": "Surat", "Salary": 70000, "Rating": 4.1},
    {"Name": "Bhumi", "Department": "HR", "City": "Pune", "Salary": 53000, "Rating": 3.9},
    {"Name": "Chetan", "Department": "Finance", "City": "Surat", "Salary": 64000, "Rating": 3.7},
    {"Name": "Devika", "Department": "IT", "City": "Mumbai", "Salary": 86000, "Rating": 4.5},
    {"Name": "Esha", "Department": "Sales", "City": "Surat", "Salary": 45000, "Rating": 3.4},
]
people = sc.createDataFrame([Row(**d) for d in data])
people.printSchema()
people.show()

tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, "team.parquet")

people.write.mode("overwrite").parquet(path)
print("Parquet written to:", path)
print("Files inside      :", [f for f in os.listdir(path) if not f.startswith(".")])

df_read = sc.read.parquet(path)
df_read.printSchema()
df_read.show()

print("Schema preserved :", people.schema == df_read.schema)
print(f"Row count original = {people.count()} | reloaded = {df_read.count()} | equal = {people.count() == df_read.count()}")

df_read.select("Rating").show()

shutil.rmtree(tmp_dir)
sc.stop()
print("Cleaned up.")

root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+------+----------+------+------+------+
|  Name|Department|  City|Salary|Rating|
+------+----------+------+------+------+
| Aryan|        IT| Surat| 70000|   4.1|
| Bhumi|        HR|  Pune| 53000|   3.9|
|Chetan|   Finance| Surat| 64000|   3.7|
|Devika|        IT|Mumbai| 86000|   4.5|
|  Esha|     Sales| Surat| 45000|   3.4|
+------+----------+------+------+------+

Parquet written to: /tmp/tmpzr1vinmh/team.parquet
Files inside      : ['part-00001-7233f82d-9833-4b53-b1ae-c4b311360942-c000.snappy.parquet', '_SUCCESS', 'part-00000-7233f82d-9833-4b53-b1ae-c4b311360942-c000.snappy.parquet']
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+------+---------